#### READ THE ORDER REVIEWS DATA FROM VOLUME

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/OrdersReviews/'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/order_reviews'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERS_REVIEWS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
order_reviews_schema = """review_id STRING, 
                 order_id STRING, 
                 review_score INT, 
                 review_comment_title STRING, 
                 review_comment_message STRING"""

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.ORDERS_REVIEWS`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

order_reviews_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(order_reviews_schema)
                 .load(f'{BASE_DIR}')
        )
order_reviews_trans_df = order_reviews_df.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

# order_reviews_trans_df.printSchema()
# order_reviews_trans_df.limit(1).display()

qry = (
    order_reviews_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()